In [5]:
import os
import pandas as pd
import gc
import re
import ast
from pathlib import Path

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:

TRAIN_PATH = "/content/drive/MyDrive/ASQP/data/ate_asqp_train.csv"
TEST_PATH = "/content/drive/MyDrive/ASQP/data/ate_asqp_test.csv"

DATA = "/content/drive/MyDrive/ASQP/data/"
FINAL_OUT_DIR = Path("/content/drive/MyDrive/ASQP/final_raw")
FINAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(
    TRAIN_PATH,
    sep=";",
    quotechar='"',
    engine="python",
    encoding="utf-8",
    on_bad_lines="skip"
)

df_raw_test = pd.read_csv(
    TEST_PATH,
    sep=";",
    quotechar='"',
    engine="python",
    encoding="utf-8",
    on_bad_lines="skip"
)


TEXT_COL = "text"
TARGET_COL = "target"



In [ ]:
df_raw.columns = [c.strip() for c in df_raw.columns]

print("Colunas encontradas:")
print(df_raw.columns.tolist())

df = df_raw.drop('target_aspects', axis=1)

required_cols = ["text", "target_quadruples"]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Colunas ausentes: {missing}")

# cria IDs
if "id" not in df.columns:
  df.insert(0,"id",[f"ex_{i:05d}" for i in range(len(df))])

if df["id"].duplicated().any():
  raise ValueError("Existem IDs duplicados no dataset.")

print(df[["id", "text", "target_quadruples"]].head(4))

print(df.columns.tolist())

Colunas encontradas:
['text', 'target_aspects', 'target_quadruples']
         id                                               text  \
0  ex_00000  Hotel com condições gerais muito más. Infraest...   
1  ex_00001  O hotel segue corretamente os padrões do Mercu...   
2  ex_00002  O hotel é muito luxuoso e em frente o central ...   
3  ex_00003  O hotel em geral é bom, o que achamos ruim foi...   

                                   target_quadruples  
0  [{'category': 'general', 'aspect': {'term': 'H...  
1  [{'category': 'location', 'aspect': {'term': '...  
2  [{'category': 'general', 'aspect': {'term': 'h...  
3  [{'category': 'general', 'aspect': {'term': 'h...  
['id', 'text', 'target_quadruples']


In [ ]:
def clean_text(text):
  if pd.isna(text):
    return ""

  text = str(text)
  text = text.lower()
  text = re.sub(r"[^a-záàâãéêíóôõúç\s]", " ", text)
  text = re.sub(r"\d+", " ", text)
  text = re.sub(r"\s+", " ", text).strip()
  return text


def clean_phrase(text):
  if text is None:
    return "NULL"

  if isinstance(text, float) and pd.isna(text):
    return "NULL"

  text = clean_text(text)

  if text == "":return "NULL"

  return text

In [ ]:
def parse_quadruples(x):
  if pd.isna(x):
    return []

  if isinstance(x, list):
    return x

  x = str(x).strip()

  if x == "" or x == "[]":
    return []

  try:
    parsed = ast.literal_eval(x)

    if isinstance(parsed, list):
      return parsed

    return []

  except Exception as e:
      print("Erro ao converter target_quadruples:")
      print(x[:500])
      print("Erro:", e)
      return []


def target_from_quads(quads):
  targets = []

  for q in quads:
    if not isinstance(q, dict):
      continue

    aspect = q.get("aspect", {})
    sentiment = q.get("sentiment", {})

    if isinstance(aspect, dict):
      aspect_term = clean_phrase(aspect.get("term", "NULL"))
    else:
      aspect_term = clean_phrase(aspect)

    category = clean_phrase(q.get("category", "NULL"))

    if isinstance(sentiment, dict):
      opinion_term = clean_phrase(sentiment.get("term", "NULL"))
    else:
      opinion_term = clean_phrase(sentiment)

    polarity = clean_phrase(q.get("polarity", "NULL"))

    targets.append(f"[A] {aspect_term} [C] {category} [O] {opinion_term} [P] {polarity}")

  return " [SSEP] ".join(targets)

In [ ]:
records = []

for _, row in df.iterrows():
  sample_id = str(row["id"])
  text = clean_text(row["text"])

  quads = parse_quadruples(row["target_quadruples"])
  target = target_from_quads(quads)

  records.append({
        "id": sample_id,
        "text": text,
        "input": "extrair quadruplas:" +text,
        "target": target,
        "quadruples": quads,
        "augmented": False,
        "augmentation": "none",
        "source_id": sample_id})

train_df = pd.DataFrame(records)

print("Amostras:", len(train_df))
print("Inputs vazios:", (train_df["text"] == "").sum())
print("Targets vazios:", (train_df["target"] == "").sum())

train_df[["id", "text","input", "target"]].head()

Amostras: 730
Inputs vazios: 0
Targets vazios: 0


,id,text,input,target
0,ex_00000,hotel com condições gerais muito más infraestr...,extrair quadruplas:hotel com condições gerais ...,[A] hotel [C] general [O] más [P] neg [SSEP] [...
1,ex_00001,o hotel segue corretamente os padrões do mercu...,extrair quadruplas:o hotel segue corretamente ...,[A] localização [C] location [O] excelente [P]...
2,ex_00002,o hotel é muito luxuoso e em frente o central ...,extrair quadruplas:o hotel é muito luxuoso e e...,[A] hotel [C] general [O] luxuoso [P] pos [SSE...
3,ex_00003,o hotel em geral é bom o que achamos ruim foi ...,extrair quadruplas:o hotel em geral é bom o qu...,[A] hotel [C] general [O] bom [P] pos [SSEP] [...
4,ex_00004,em cidades grandes e com muitas atrações como ...,extrair quadruplas:em cidades grandes e com mu...,[A] hotel [C] location [O] próximo do louvre d...


In [ ]:
records = []

for _, row in df_raw_test.iterrows():
  sample_id = str(row["id"])
  text = clean_text(row["text"])

  quads = parse_quadruples(row["target_quadruples"])
  target = target_from_quads(quads)

  records.append({
        "id": sample_id,
        "input": "extrair quadruplas:" +text,
        "target": target,
  })

test_df = pd.DataFrame(records)

print("Amostras:", len(test_df))
print("Targets vazios:", (test_df["target"] == "").sum())

test_df[["id","input", "target"]].head()

Amostras: 253
Targets vazios: 0


,id,input,target
0,549,extrair quadruplas:o hotel diz que foi recente...,[A] pia [C] structure [O] entupida [P] neg [SS...
1,1444,extrair quadruplas:hotel muito agradáve próxim...,[A] hotel [C] general [O] agradáve [P] pos [SS...
2,265,extrair quadruplas:seviço péssimo muito caro a...,[A] seviço [C] service [O] péssimo [P] neg [SS...
3,300,extrair quadruplas:você só deve se hospedar ne...,[A] quarto [C] structure [O] vaso sanitário fi...
4,715,extrair quadruplas:hotel com ótima localização...,[A] localização [C] location [O] ótima [P] pos...


In [ ]:
# Salva os df pre processados

SAVE_DIR = Path("/content/drive/MyDrive/ASQP/data_processado")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(
    SAVE_DIR / "train_processed.csv",
    index=False,
    encoding="utf-8"
)

test_df.to_csv(
    SAVE_DIR / "test_processed.csv",
    index=False,
    encoding="utf-8"
)

print("Train salvo em:")
print(SAVE_DIR / "train_processed.csv")

print("\nTest salvo em:")
print(SAVE_DIR / "test_processed.csv")

Train salvo em:
/content/drive/MyDrive/ASQP/data_processado/train_processed.csv

Test salvo em:
/content/drive/MyDrive/ASQP/data_processado/test_processed.csv
